In [ ]:
# imports
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

COLORS = {
    "dp": "#2196F3", "tp": "#4CAF50", "pp": "#FF9800",
    "mixed": "#9C27B0", "schedule": "#E91E63",
}


In [ ]:
# settings
BASE = os.getcwd()
MODEL_NAME = "llama_1b"
DATA_DIR = os.path.join(BASE, "power_logs", MODEL_NAME)
PLOTS_DIR = os.path.join(BASE, "plots", MODEL_NAME)
os.makedirs(PLOTS_DIR, exist_ok=True)
print(f"Reading power logs from: {DATA_DIR}")
print(f"Saving plots to: {PLOTS_DIR}")

INTERVAL_SECONDS = 0.5
TRAIN_ITERS = 200
GLOBAL_BATCH = 16
SEQ_LEN = 1024

A = [400, 100, 100]   # CI per period (gCO₂/kWh)
CI_FIXED = 400    # baseline constant


In [ ]:
# helpers
def load_power_csv(path):
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    if "time_step" not in df.columns:
        df.columns = ["timestamp", "total_power_W",
                      "GPU_0_W", "GPU_1_W", "GPU_2_W", "GPU_3_W"]
        df["time_step"] = 0
    return df


def get_ci_schedule_wallclock(n_samples, interval, A, schedule_duration_s):
    ci = np.full(n_samples, A[-1], dtype=float)
    period_s = schedule_duration_s / len(A)
    for i, c in enumerate(A):
        idx_start = max(0, min(int(i * period_s / interval), n_samples))
        idx_end   = max(0, min(int((i + 1) * period_s / interval), n_samples))
        ci[idx_start:idx_end] = c
    return ci


def compute_carbon(power_samples, ci_array, interval=0.5):
    energy_kwh = power_samples * interval / 3_600_000
    grams = energy_kwh * ci_array
    return np.cumsum(grams), grams.sum()


In [ ]:
# ci colors
_CI_CMAP = plt.cm.YlOrRd
_CI_NORM = None

def build_ci_norm(A):
    global _CI_NORM
    ci_min, ci_max = min(A), max(A)
    if ci_max == ci_min:
        _CI_NORM = mcolors.Normalize(vmin=ci_min * 0.9, vmax=ci_max * 1.1)
    else:
        _CI_NORM = mcolors.Normalize(vmin=ci_min, vmax=ci_max)

def ci_to_color(ci_val):
    if _CI_NORM is None:
        raise RuntimeError("Call build_ci_norm(A) first.")
    return _CI_CMAP(_CI_NORM(ci_val))

def ci_label(ci_val, A):
    ci_min, ci_max = min(A), max(A)
    if ci_val == ci_max and ci_val != ci_min:
        return f"{ci_val} g/kWh (high)"
    if ci_val == ci_min and ci_val != ci_max:
        return f"{ci_val} g/kWh (low)"
    return f"{ci_val} g/kWh"

build_ci_norm(A)

def add_ci_colorbar(fig, ax, A, label="CI (gCO₂/kWh)"):
    sm = plt.cm.ScalarMappable(cmap=_CI_CMAP, norm=_CI_NORM)
    sm.set_array([])
    cb = fig.colorbar(sm, ax=ax, orientation="vertical",
                      fraction=0.03, pad=0.02, shrink=0.75)
    cb.set_label(label, fontsize=8)
    cb.set_ticks(sorted(set(A)))
    cb.set_ticklabels([ci_label(c, A) for c in sorted(set(A))])
    return cb


def _classify(cfg):
    if cfg["is_schedule"]:
        return "schedule"
    dims_gt1 = sum(1 for d in (cfg["tp"], cfg["pp"], cfg["dp"]) if d > 1)
    if dims_gt1 >= 2: return "mixed"
    if cfg["tp"] > 1: return "tp"
    if cfg["pp"] > 1: return "pp"
    return "dp"


def _label(cfg):
    if cfg["is_schedule"]:
        return "Hybrid\nSchedule"
    parts = []
    if cfg["tp"] > 1: parts.append(f"TP={cfg['tp']}")
    if cfg["pp"] > 1: parts.append(f"PP={cfg['pp']}")
    if cfg["dp"] > 1: parts.append(f"DP={cfg['dp']}")
    return ",".join(parts) if parts else "DP=1"


def annotate_ci_periods(ax, A, schedule_duration_s, y_label_frac=0.04):
    period_s = schedule_duration_s / len(A)
    ylo, yhi = ax.get_ylim()
    y_text = ylo + (yhi - ylo) * y_label_frac
    for i, c in enumerate(A):
        x0, x1 = i * period_s, (i + 1) * period_s
        ax.axvspan(x0, x1, alpha=0.18, color=ci_to_color(c), zorder=0)
        ax.text((x0 + x1) / 2, y_text, ci_label(c, A),
                ha="center", va="bottom", fontsize=8,
                fontweight="bold", color=ci_to_color(c), alpha=0.85)
    xlim = ax.get_xlim()
    if xlim[1] > schedule_duration_s:
        ax.axvspan(schedule_duration_s, xlim[1], alpha=0.10,
                   color=ci_to_color(A[-1]), zorder=0)


In [ ]:
# load csvs compute
all_possible_configs = {
    "tp1_pp1_dp4": {"tp": 1, "pp": 1, "dp": 4, "file": "elastic_tp1_pp1_dp4_power.csv", "is_schedule": False},
    "tp4_pp1_dp1": {"tp": 4, "pp": 1, "dp": 1, "file": "elastic_tp4_pp1_dp1_power.csv", "is_schedule": False},
    "tp1_pp4_dp1": {"tp": 1, "pp": 4, "dp": 1, "file": "elastic_tp1_pp4_dp1_power.csv", "is_schedule": False},
    "tp2_pp2_dp1": {"tp": 2, "pp": 2, "dp": 1, "file": "elastic_tp2_pp2_dp1_power.csv", "is_schedule": False},
    "tp2_pp1_dp2": {"tp": 2, "pp": 1, "dp": 2, "file": "elastic_tp2_pp1_dp2_power.csv", "is_schedule": False},
    "tp1_pp2_dp2": {"tp": 1, "pp": 2, "dp": 2, "file": "elastic_tp1_pp2_dp2_power.csv", "is_schedule": False},
}

schedule_configs_def = {
    "hybrid_schedule": {
        "tp": None, "pp": None, "dp": None,
        "file": "elastic_example_schedule_power.csv",
        "is_schedule": True,
    },
}

configs = {}
skipped = []

for name, cfg in {**all_possible_configs, **schedule_configs_def}.items():
    path = os.path.join(DATA_DIR, cfg["file"])
    if not os.path.exists(path):
        skipped.append(name)
        continue
    df = load_power_csv(path)
    if len(df) < 2:
        skipped.append(name)
        continue
    warmup = int(len(df) * 0.2)
    steady = df.iloc[warmup:]
    cfg["avg_power"] = df["total_power_W"].mean()
    cfg["steady_avg_power"] = steady["total_power_W"].mean()
    cfg["peak_power"] = df["total_power_W"].max()
    cfg["df"] = df
    configs[name] = cfg
    tag = " [SCHED]" if cfg["is_schedule"] else ""
    print(f"{name}{tag}: avg={cfg['avg_power']:.1f}W  "
          f"steady={cfg['steady_avg_power']:.1f}W  samples={len(df)}")

if skipped:
    print(f"\nSkipped: {', '.join(skipped)}")

static_configs = {k: v for k, v in configs.items() if not v["is_schedule"]}
sched_configs  = {k: v for k, v in configs.items() if v["is_schedule"]}
n_configs = len(configs)
all_configs = [(_label(cfg), _classify(cfg), cfg) for _, cfg in configs.items()]


In [ ]:
# 4b derive ci
if not configs:
    raise RuntimeError("No configs loaded — cannot derive schedule duration.")

CI_SCHEDULE_DURATION_S = max(len(cfg["df"]) for cfg in configs.values()) * INTERVAL_SECONDS
slowest = max(configs.items(), key=lambda kv: len(kv[1]["df"]))[0]
print(f"\nCI schedule duration (derived): {CI_SCHEDULE_DURATION_S:.1f}s "
      f"(slowest = {slowest})")
print(f"Each of the {len(A)} periods covers "
      f"{CI_SCHEDULE_DURATION_S / len(A):.1f}s of wall-clock time.")


In [ ]:
# metrics
total_tokens = TRAIN_ITERS * GLOBAL_BATCH * SEQ_LEN

print(f"\n{'Label':20s} | {'Time':>6s} | {'Power':>7s} | "
      f"{'Energy':>7s} | {'C_fix':>7s} | {'C_var':>7s} | {'Tput':>9s}")
print("-" * 82)

for label, _, cfg in all_configs:
    power = cfg["df"]["total_power_W"].to_numpy()
    n = len(power)
    total_time_s = n * INTERVAL_SECONDS
    cfg["train_time_s"] = total_time_s
    cfg["total_time_s"] = total_time_s
    cfg["avg_power_w"] = power.mean()
    cfg["total_energy_wh"] = power.sum() * INTERVAL_SECONDS / 3600
    cfg["tokens_per_sec"] = total_tokens / total_time_s

    _, cfg["carbon_fixed_g"] = compute_carbon(power, np.full(n, CI_FIXED))
    ci_var = get_ci_schedule_wallclock(n, INTERVAL_SECONDS, A, CI_SCHEDULE_DURATION_S)
    _, cfg["carbon_varying_g"] = compute_carbon(power, ci_var)

    tag = " ★" if cfg["is_schedule"] else ""
    print(f"{label.replace(chr(10),' '):20s} | {total_time_s:5.1f}s | "
          f"{cfg['avg_power_w']:6.1f}W | {cfg['total_energy_wh']:6.2f}Wh | "
          f"{cfg['carbon_fixed_g']:5.1f}g | {cfg['carbon_varying_g']:5.1f}g | "
          f"{cfg['tokens_per_sec']:7.1f} t/s{tag}")


In [ ]:
# power traces config
if n_configs > 0:
    ncols = min(3, n_configs)
    nrows = (n_configs + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(12, 2.8 * nrows))
    axes = np.atleast_1d(axes).flatten()

    for ax, (name, cfg) in zip(axes, configs.items()):
        df = cfg["df"]
        warmup_idx = int(len(df) * 0.2)
        x = np.arange(len(df)) * INTERVAL_SECONDS
        y = df["total_power_W"].to_numpy()

        ax.plot(x[:warmup_idx], y[:warmup_idx], color="#B0BEC5", linewidth=1.4)
        ax.plot(x[warmup_idx:], y[warmup_idx:],
                color=COLORS["schedule"] if cfg["is_schedule"] else "#1976D2",
                linewidth=1.6)
        ax.axhline(cfg["steady_avg_power"], color="#43A047", linestyle=":", linewidth=1.2)

        if cfg["is_schedule"]:
            title = "Hybrid Schedule"
            steps = df["time_step"].to_numpy()
            for j in range(1, len(steps)):
                if steps[j] < steps[j-1] and steps[j-1] > 0:
                    ax.axvline(x[j], color="#FF9800", linestyle="-.", linewidth=0.8, alpha=0.7)
        else:
            title = f"TP={cfg['tp']}, PP={cfg['pp']}, DP={cfg['dp']}"

        ax.set_title(title, fontsize=9, fontweight="bold")
        ax.set_xlabel("Time (s)", fontsize=8)
        ax.set_ylabel("Power (W)", fontsize=8)
        ax.tick_params(labelsize=7)
        ax.text(0.02, 0.92, f"avg={cfg['steady_avg_power']:.0f}W",
                transform=ax.transAxes, fontsize=7, color="#2E7D32")

    for ax in axes[n_configs:]:
        ax.set_visible(False)

    fig.suptitle("Power Consumption Per Configuration", fontsize=11, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, "graph0_power_trends.png"), dpi=150, bbox_inches="tight")
    plt.show()


In [ ]:
# fixed vs varying
print("\n" + "=" * 72)
print(f"CI SCHEDULE:  A = {A} gCO₂/kWh over {CI_SCHEDULE_DURATION_S:.0f}s")
period_s = CI_SCHEDULE_DURATION_S / len(A)
for i, c in enumerate(A):
    print(f"  Period {i+1}: {i*period_s:.0f}–{(i+1)*period_s:.0f}s → {ci_label(c, A)}")
print(f"Fixed baseline: {CI_FIXED} gCO₂/kWh")
print("=" * 72)

if n_configs > 0:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

    for ax, use_varying in zip(axes, [False, True]):
        for name, cfg in configs.items():
            power = cfg["df"]["total_power_W"].to_numpy()
            n = len(power)
            ci = (get_ci_schedule_wallclock(n, INTERVAL_SECONDS, A, CI_SCHEDULE_DURATION_S)
                  if use_varying else np.full(n, CI_FIXED))
            cum, _ = compute_carbon(power, ci)
            x = np.arange(n) * INTERVAL_SECONDS

            style = {"linewidth": 2.2, "linestyle": "--"} if cfg["is_schedule"] else {"linewidth": 1.6}
            lbl = "Hybrid Schedule" if cfg["is_schedule"] else \
                  f"TP={cfg['tp']},PP={cfg['pp']},DP={cfg['dp']}"
            color = COLORS["schedule"] if cfg["is_schedule"] else None
            ax.plot(x, cum, label=lbl, color=color, **style)
            ax.scatter(x[-1], cum[-1], s=35, zorder=3, color=color)

        if use_varying:
            annotate_ci_periods(ax, A, CI_SCHEDULE_DURATION_S)
            ax.set_title(f"Varying CI  (A={A}, {CI_SCHEDULE_DURATION_S:.0f}s)",
                         fontsize=10, fontweight="bold")
            add_ci_colorbar(fig, ax, A)
        else:
            ax.set_title(f"Fixed CI  ({CI_FIXED} gCO₂/kWh)", fontsize=10, fontweight="bold")

        ax.set_xlabel("Time (s)", fontsize=9)
        ax.set_ylabel("Cumulative CO₂ (g)", fontsize=9)
        ax.legend(fontsize=6.5, ncol=2, frameon=False)

    plt.suptitle("Impact of Time-Varying Carbon Intensity",
                 fontsize=11, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, "graph_carbon_varying_ci.png"),
                dpi=150, bbox_inches="tight")
    plt.show()


In [ ]:
# compact bar chart
patches = [
    mpatches.Patch(color=COLORS["dp"],       label="Data Parallel"),
    mpatches.Patch(color=COLORS["tp"],       label="Tensor Parallel"),
    mpatches.Patch(color=COLORS["pp"],       label="Pipeline Parallel"),
    mpatches.Patch(color=COLORS["mixed"],    label="Mixed"),
    mpatches.Patch(color=COLORS["schedule"], label="Hybrid Schedule"),
]

if all_configs:
    labels_bar = [c[0].replace("\n", " ") for c in all_configs]
    colors_bar = [COLORS[c[1]] for c in all_configs]
    time_vals      = [c[2]["train_time_s"]     for c in all_configs]
    energy_vals    = [c[2]["total_energy_wh"]  for c in all_configs]
    carbon_fixed   = [c[2]["carbon_fixed_g"]   for c in all_configs]
    carbon_varying = [c[2]["carbon_varying_g"] for c in all_configs]
    is_sched       = [c[2]["is_schedule"]      for c in all_configs]

    edge_colors = ["#E91E63" if s else "white" for s in is_sched]
    edge_widths = [2.5 if s else 0.5 for s in is_sched]

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    # panel 1: total time
    ax = axes[0]
    ax.bar(labels_bar, time_vals, color=colors_bar,
           edgecolor=edge_colors, linewidth=edge_widths)
    for i, v in enumerate(time_vals):
        ax.text(i, v + max(time_vals) * 0.01, f"{v:.0f}s",
                ha="center", va="bottom", fontsize=7)
    ax.set_ylabel("Time (s)", fontsize=9)
    ax.set_title("Total Training Time", fontsize=10, fontweight="bold")
    ax.tick_params(axis="x", labelsize=7, rotation=20)

    # panel 2: total energy
    ax = axes[1]
    ax.bar(labels_bar, energy_vals, color=colors_bar,
           edgecolor=edge_colors, linewidth=edge_widths)
    for i, v in enumerate(energy_vals):
        ax.text(i, v + max(energy_vals) * 0.01, f"{v:.1f}",
                ha="center", va="bottom", fontsize=7)
    ax.set_ylabel("Energy (Wh)", fontsize=9)
    ax.set_title("Total Energy", fontsize=10, fontweight="bold")
    ax.tick_params(axis="x", labelsize=7, rotation=20)

    # panel 3: carbon fixed vs varying
    ax = axes[2]
    x = np.arange(len(labels_bar))
    width = 0.38
    ax.bar(x - width/2, carbon_fixed, width, color=colors_bar,
           alpha=0.5, edgecolor=edge_colors, linewidth=edge_widths,
           label=f"Fixed ({CI_FIXED})")
    ax.bar(x + width/2, carbon_varying, width, color=colors_bar,
           edgecolor="black", linewidth=0.6,
           label=f"Varying (A={A})")
    for xi, vv in zip(x, carbon_varying):
        ax.text(xi + width/2, vv + max(carbon_varying) * 0.01,
                f"{vv:.0f}", ha="center", va="bottom", fontsize=6.5)
    ax.set_ylabel("Carbon (gCO₂)", fontsize=9)
    ax.set_title("Total Carbon — Fixed vs Varying CI", fontsize=10, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels(labels_bar, fontsize=7, rotation=20)
    ax.legend(fontsize=7, loc="upper left")

    fig.legend(handles=patches, fontsize=8, ncol=5,
               loc="upper center", bbox_to_anchor=(0.5, 1.03), frameon=False)
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, "graph_metrics_overview.png"),
                dpi=150, bbox_inches="tight")
    plt.show()


In [ ]:
# schedule detail power
for name, cfg in sched_configs.items():
    df = cfg["df"]
    x = np.arange(len(df)) * INTERVAL_SECONDS
    y = df["total_power_W"].to_numpy()
    steps = df["time_step"].to_numpy()

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 5), sharex=True,
                                    gridspec_kw={"height_ratios": [3, 1]})

    ax1.plot(x, y, color=COLORS["schedule"], linewidth=1.3, alpha=0.85)
    ax1.fill_between(x, y, alpha=0.15, color=COLORS["schedule"])
    ax1.set_ylabel("Power (W)", fontsize=9)
    ax1.set_title("Hybrid Schedule — Power Trace with Phase Boundaries",
                  fontsize=11, fontweight="bold")

    phase_colors = ["#E3F2FD", "#E8F5E9", "#FFF3E0", "#F3E5F5",
                    "#FCE4EC", "#E0F7FA", "#FFF9C4"]
    boundaries = [0]
    for j in range(1, len(steps)):
        if steps[j] < steps[j-1] and steps[j-1] > 0:
            boundaries.append(j)
    boundaries.append(len(df))

    for p_idx in range(len(boundaries) - 1):
        s_idx = boundaries[p_idx]
        e_idx = boundaries[p_idx + 1]
        if s_idx < len(x) and e_idx <= len(x):
            ax1.axvspan(x[s_idx], x[min(e_idx - 1, len(x) - 1)],
                        alpha=0.15, color=phase_colors[p_idx % len(phase_colors)])
            mid_x = (x[s_idx] + x[min(e_idx - 1, len(x) - 1)]) / 2
            ax1.text(mid_x, ax1.get_ylim()[1] * 0.95, f"P{p_idx + 1}",
                     ha="center", va="top", fontsize=8, fontweight="bold", alpha=0.7)

    ax2.plot(x, steps, color="#666", linewidth=1.2)
    ax2.set_xlabel("Time (s)", fontsize=9)
    ax2.set_ylabel("Step", fontsize=9)

    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, "graph_schedule_detailed.png"),
                dpi=150, bbox_inches="tight")
    plt.show()


In [ ]:
# summary
if all_configs:
    summary_rows = []
    for label, _, cfg in all_configs:
        summary_rows.append({
            "Config":           label.replace("\n", " "),
            "Type":             "Schedule" if cfg["is_schedule"] else "Static",
            "Time(s)":          round(cfg["train_time_s"], 1),
            "Avg Power(W)":     round(cfg["avg_power_w"], 1),
            "Energy(Wh)":       round(cfg["total_energy_wh"], 2),
            f"C_Fixed@{CI_FIXED}(g)": round(cfg["carbon_fixed_g"], 2),
            "C_Vary(g)":        round(cfg["carbon_varying_g"], 2),
            "Tput(tok/s)":      round(cfg["tokens_per_sec"], 1),
        })

    summary_df = pd.DataFrame(summary_rows)
    print("\n" + "=" * 95)
    print("SUMMARY TABLE")
    print("=" * 95)
    print(summary_df.to_string(index=False))
    summary_df.to_csv(os.path.join(PLOTS_DIR, "summary_table.csv"), index=False)
    print(f"\nSaved: {os.path.join(PLOTS_DIR, 'summary_table.csv')}")

In [ ]:
# model savings
MODEL_PARAMS_M = {
    "gpt2":         124,
    "gpt2_medium":  355,
    "gpt2_large":   774,
    "llama_1b":    1100,
    "llama_2b":    2000,
    "llama_3b":    3000,
}

FAMILIES = {
    "gpt2":  {
        "models": ["gpt2", "gpt2_medium", "gpt2_large"],
        "A":      [600, 300, 300],
        "title":  "GPT-2 family",
    },
    "llama": {
        "models": ["llama_1b", "llama_2b", "llama_3b"],
        "A":      [400, 100, 100],
        "title":  "Llama family",
    },
}

STATIC_FILES = [
    "elastic_tp1_pp1_dp4_power.csv",
    "elastic_tp4_pp1_dp1_power.csv",
    "elastic_tp1_pp4_dp1_power.csv",
    "elastic_tp2_pp2_dp1_power.csv",
    "elastic_tp2_pp1_dp2_power.csv",
    "elastic_tp1_pp2_dp2_power.csv",
]
SCHEDULE_FILE = "elastic_example_schedule_power.csv"

CROSS_PLOTS_DIR = os.path.join(BASE, "plots", "cross_model")
os.makedirs(CROSS_PLOTS_DIR, exist_ok=True)


def _load_runs(ddir, files):
    out = {}
    for f in files:
        p = os.path.join(ddir, f)
        if not os.path.exists(p):
            continue
        df = load_power_csv(p)
        if len(df) >= 2:
            out[f] = df
    return out


def _carbon_varying(df, A_family, schedule_duration_s):
    power = df["total_power_W"].to_numpy()
    ci = get_ci_schedule_wallclock(len(power), INTERVAL_SECONDS,
                                   A_family, schedule_duration_s)
    _, total = compute_carbon(power, ci, INTERVAL_SECONDS)
    return total


def _collect_family_rows(models, A_family):
    rows = []
    for m in sorted(models, key=lambda x: MODEL_PARAMS_M.get(x, 0)):
        ddir = os.path.join(BASE, "power_logs", m)
        if not os.path.isdir(ddir):
            print(f"[skip] {m}: no directory")
            continue

        static_runs = _load_runs(ddir, STATIC_FILES)
        sched_runs  = _load_runs(ddir, [SCHEDULE_FILE])
        if not static_runs or not sched_runs:
            print(f"[skip] {m}: missing runs")
            continue

        all_runs = {**static_runs, **sched_runs}
        sched_dur_s = max(len(df) for df in all_runs.values()) * INTERVAL_SECONDS

        static_carbons = [_carbon_varying(df, A_family, sched_dur_s)
                          for df in static_runs.values()]
        sched_carbon   = _carbon_varying(next(iter(sched_runs.values())),
                                         A_family, sched_dur_s)

        best_fixed = min(static_carbons)
        saving_pct = (best_fixed - sched_carbon) / best_fixed * 100

        rows.append({
            "model":        m,
            "params_M":     MODEL_PARAMS_M.get(m, 0),
            "sched_dur_s":  sched_dur_s,
            "best_fixed_g": best_fixed,
            "schedule_g":   sched_carbon,
            "saving_pct":   saving_pct,
        })
        print(f"{m:15s} | window={sched_dur_s:6.0f}s | "
              f"best fixed={best_fixed:7.1f}g | schedule={sched_carbon:7.1f}g | "
              f"saved={saving_pct:+5.1f}%")
    return rows


def _plot_family(rows, A_family, title, outfile):
    if len(rows) < 2:
        print(f"[skip plot] {title}: need ≥2 models, got {len(rows)}")
        return
    params  = [r["params_M"]    for r in rows]
    savings = [r["saving_pct"]  for r in rows]
    names   = [r["model"]       for r in rows]
    windows = [r["sched_dur_s"] for r in rows]

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(params, savings, "o-", color=COLORS["schedule"],
            linewidth=2.5, markersize=11,
            markerfacecolor="white", markeredgewidth=2.5)
    for p, s, n, w in zip(params, savings, names, windows):
        ax.annotate(f"{n}\n{s:+.1f}%  ({w:.0f}s)", (p, s),
                    textcoords="offset points", xytext=(8, 6), fontsize=8)
    ax.axhline(0, color="gray", linestyle=":", alpha=0.6)
    ax.set_xscale("log")
    ax.set_xlabel("Model size (parameters, millions)", fontsize=11)
    ax.set_ylabel("Carbon saved by elastic schedule (%)", fontsize=11)
    ax.set_title(
        f"{title} — elastic savings vs model size\n"
        f"CI schedule A = {A_family}, window auto-sized per model",
        fontsize=11, fontweight="bold",
    )
    plt.tight_layout()
    plt.savefig(os.path.join(CROSS_PLOTS_DIR, outfile),
                dpi=150, bbox_inches="tight")
    plt.show()


In [ ]:
# run family
for family_key, fam in FAMILIES.items():
    print(f"\n── {fam['title']}  (A = {fam['A']}) ──")
    rows = _collect_family_rows(fam["models"], fam["A"])
    _plot_family(rows, fam["A"], fam["title"],
                 f"savings_vs_model_size_{family_key}.png")

In [ ]:
# power landscape
CONFIG_ORDER = [
    ("elastic_tp1_pp1_dp4_power.csv", "DP=4"),
    ("elastic_tp4_pp1_dp1_power.csv", "TP=4"),
    ("elastic_tp1_pp4_dp1_power.csv", "PP=4"),
    ("elastic_tp2_pp2_dp1_power.csv", "TP=2,PP=2"),
    ("elastic_tp2_pp1_dp2_power.csv", "TP=2,DP=2"),
    ("elastic_tp1_pp2_dp2_power.csv", "PP=2,DP=2"),
]

POWER_LANDSCAPE_FAMILIES = {
    "GPT-2":  ["gpt2", "gpt2_medium", "gpt2_large"],
    "Llama":  ["llama_1b", "llama_2b", "llama_3b"],
    "Pythia": ["pythia_410m", "pythia_1_4b", "pythia_2_8b"],
}

SIZE_LABELS = {
    "gpt2": "small (124M)", "gpt2_medium": "medium (355M)", "gpt2_large": "large (774M)",
    "llama_1b": "1B", "llama_2b": "2B", "llama_3b": "3B",
    "pythia_410m": "410M", "pythia_1_4b": "1.4B", "pythia_2_8b": "2.8B",
}

FAMILY_CMAPS = {"GPT-2": plt.cm.Blues, "Llama": plt.cm.Greens, "Pythia": plt.cm.Purples}


def _steady_avg_power(path, warmup_frac=0.2):
    """Steady-state mean total power (W). Returns None if file missing/too short."""
    if not os.path.exists(path):
        return None
    df = load_power_csv(path)
    if len(df) < 5:
        return None
    warmup = int(len(df) * warmup_frac)
    return df["total_power_W"].iloc[warmup:].mean()


In [ ]:
# collect family model
landscape_data = {}
for fam_name, models in POWER_LANDSCAPE_FAMILIES.items():
    fam = {}
    for m in models:
        ddir = os.path.join(BASE, "power_logs", m)
        if not os.path.isdir(ddir):
            continue
        powers = [_steady_avg_power(os.path.join(ddir, fn))
                  for fn, _ in CONFIG_ORDER]
        if any(p is not None for p in powers):
            fam[m] = powers
    if fam:
        landscape_data[fam_name] = fam


In [ ]:
# plot panel family
if not landscape_data:
    print("No power logs found for any family — nothing to plot.")
else:
    n_fam = len(landscape_data)
    fig, axes = plt.subplots(1, n_fam, figsize=(5.0 * n_fam, 4.2))
    axes = np.atleast_1d(axes)

    cfg_labels = [lbl for _, lbl in CONFIG_ORDER]
    x = np.arange(len(cfg_labels))

    for ax, (fam_name, fam) in zip(axes, landscape_data.items()):
        sizes = list(fam.keys())
        n_sizes = len(sizes)
        bar_w = 0.8 / max(n_sizes, 1)
        cmap = FAMILY_CMAPS.get(fam_name, plt.cm.Greys)
        shades = [cmap(0.45 + 0.45 * i / max(n_sizes - 1, 1)) for i in range(n_sizes)]

        for i, m in enumerate(sizes):
            powers = fam[m]
            offset = (i - (n_sizes - 1) / 2) * bar_w
            valid = [(xi + offset, p) for xi, p in zip(x, powers) if p is not None]
            if not valid:
                continue
            xs, ps = zip(*valid)
            ax.bar(xs, ps, bar_w, color=shades[i], edgecolor="white",
                   linewidth=0.5, label=SIZE_LABELS.get(m, m))

        ax.set_title(fam_name, fontsize=11, fontweight="bold")
        ax.set_xticks(x)
        ax.set_xticklabels(cfg_labels, fontsize=8, rotation=15, ha="right")
        ax.set_ylabel("Steady-state avg power (W)", fontsize=9)
        ax.tick_params(axis="y", labelsize=8)
        ax.legend(fontsize=7, frameon=False, loc="best")
        ax.grid(axis="y", alpha=0.3)

    fig.suptitle(
        "Power landscape across (DP, TP, PP) configurations",
        fontsize=11, fontweight="bold", y=1.02,
    )
    plt.tight_layout()

    out_dir = os.path.join(BASE, "plots", "cross_model")
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, "power_landscape_by_family.png")
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"\nSaved: {out_path}")

# compact table for
    header = f"{'Family':8s} | {'Size':18s} | " + " | ".join(f"{lbl:>10s}" for lbl in cfg_labels)
    print("\n" + header)
    print("-" * len(header))
    for fam_name, fam in landscape_data.items():
        for m, powers in fam.items():
            cells = " | ".join(f"{p:10.1f}" if p is not None else f"{'—':>10s}" for p in powers)
            print(f"{fam_name:8s} | {SIZE_LABELS.get(m, m):18s} | {cells}")

In [ ]:
# online run
# actually saw
import json


In [ ]:
# run specific paths
ONLINE_RUN_DIR   = "/bgfs/stephenlee/wat29/scheduling-carbon-aware-3d-parallelism/runs/online_20260506_161126_1736014"
ONLINE_POWER_CSV = "/bgfs/stephenlee/wat29/scheduling-carbon-aware-3d-parallelism/power_logs/gpt2_medium/online_elastic_calibration_gpt2_medium_4gpu_power.csv"
CARBON_CSV       = "/bgfs/stephenlee/wat29/scheduling-carbon-aware-3d-parallelism/data/carbon_synthetic_dataset.csv"


In [ ]:
# load everything
online_df = load_power_csv(ONLINE_POWER_CSV)
power      = online_df["total_power_W"].to_numpy()
time_step  = online_df["time_step"].to_numpy()
n_samples  = len(online_df)
wall_time  = np.arange(n_samples) * INTERVAL_SECONDS

with open(os.path.join(ONLINE_RUN_DIR, "phases.json")) as f:
    phases = json.load(f)
with open(os.path.join(ONLINE_RUN_DIR, "elastic_results.json")) as f:
    results = json.load(f)
ci_hourly      = pd.read_csv(CARBON_CSV)["carbon"].to_numpy()
BUCKET_SECONDS = results.get('seconds_per_forecast_bucket', 3600.0)

training_start_idx = next((i for i, ts in enumerate(time_step) if ts > 0), 0)
training_start_t   = training_start_idx * INTERVAL_SECONDS

# phase wall time spans relative to powermonitor t=0.
phase_walls = [training_start_t]
for p in phases:
    phase_walls.append(phase_walls[-1] + p['reconfig_s'] + p['train_s'] + p['save_s'])

ci_trace = np.array([
    ci_hourly[min(int(t // BUCKET_SECONDS), len(ci_hourly) - 1)]
    for t in wall_time
])

# cumulative carbon
cum_g, total_g = compute_carbon(power, ci_trace, INTERVAL_SECONDS)


In [ ]:
# 3 panel plot
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(11, 9), sharex=True,
                                     gridspec_kw={"height_ratios": [3, 1.2, 2]})

ax1.plot(wall_time, power, color=COLORS["schedule"], linewidth=1.2, alpha=0.85)
ax1.fill_between(wall_time, 0, power, alpha=0.12, color=COLORS["schedule"])
ax1.set_ylabel("Cluster Power (W)", fontsize=9)
ax1.set_title(
    f"Online Elastic Run — {results['total_steps']} steps, {len(phases)} phases "
    f"(deadline={results['deadline_s']:.0f}s, "
    f"switch_time={results['switch_time_s']:.1f}s, "
    f"decide_every={results['decide_every']})",
    fontsize=11, fontweight="bold",
)

if training_start_t > 0:
    ax1.axvspan(0, training_start_t, alpha=0.15, color="#9E9E9E")
    ax1.text(training_start_t / 2, ax1.get_ylim()[1] * 0.5,
             "init", ha="center", va="center",
             fontsize=8, fontweight="bold", color="#666", alpha=0.8)

# phase shading + (config, steps, observed power) labels
phase_palette = ["#E3F2FD", "#E8F5E9", "#FFF3E0", "#F3E5F5",
                 "#FCE4EC", "#E0F7FA", "#FFF9C4"]
for i, p in enumerate(phases):
    x0, x1 = phase_walls[i], min(phase_walls[i + 1], wall_time[-1])
    ax1.axvspan(x0, x1, alpha=0.30, color=phase_palette[i % len(phase_palette)])
    mid_x   = (x0 + x1) / 2
    pow_obs = p['observed_power_w'] if p.get('observed_power_w') else 0
    label = (f"P{i+1}\n"
             f"DP={p['dp']} TP={p['tp']} PP={p['pp']}\n"
             f"{p['num_steps']} steps  ~{pow_obs:.0f}W")
    ax1.text(mid_x, ax1.get_ylim()[1] * 0.93, label,
             ha="center", va="top", fontsize=8, fontweight="bold", alpha=0.85)

# panel 2: ci trace (real, time varying)
ax2.plot(wall_time, ci_trace, color="#FF6F00", linewidth=1.5)
ax2.fill_between(wall_time, 0, ci_trace, alpha=0.18, color="#FF6F00")
ax2.set_ylabel("CI (gCO₂/kWh)", fontsize=9)
ax2.tick_params(labelsize=8)

# panel 3: cumulative carbon
ax3.plot(wall_time, cum_g, color="#1976D2", linewidth=1.8)
ax3.fill_between(wall_time, 0, cum_g, alpha=0.15, color="#1976D2")
ax3.set_ylabel("Cumulative CO₂ (g)", fontsize=9)
ax3.set_xlabel("Wall time (s)", fontsize=9)
ax3.text(0.98, 0.08, f"Total: {total_g:.2f} g",
         transform=ax3.transAxes, ha="right", va="bottom",
         fontsize=10, fontweight="bold", color="#1976D2",
         bbox=dict(boxstyle="round", facecolor="white",
                   edgecolor="#1976D2", alpha=0.85))

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "graph_online_elastic_run.png"),
            dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# summary
print("\n" + "=" * 90)
print(f"ONLINE ELASTIC RUN")
print("=" * 90)
print(f"  Total wall:         {results['total_wall_s']:.1f}s "
      f"(training {results['total_train_s']:.1f}s, "
      f"reconfig {results['total_reconfig_s']:.1f}s, "
      f"checkpoint {results['total_save_s']:.1f}s)")
print(f"  Total carbon:       {total_g:.2f} gCO₂   "
      f"(CI mean during run: {ci_trace.mean():.0f} g/kWh)")
print(f"  Switch cost:        {results['switch_time_s']:.1f}s @ "
      f"{results['switch_power_w']:.0f}W")
print(f"  Decision cadence:   every {results['decide_every']} steps  "
      f"(lookahead {results['lookahead_steps']} steps)")
print(f"  Phases scheduled:   {len(phases)}")

print(f"\n  {'Phase':<6} {'Config':<22} {'Steps':>8} {'Train':>9} "
      f"{'Iter':>11} {'P(W)':>8} {'Loss':>10}")
print("  " + "-" * 80)
for p in phases:
    cfg_str = f"dp{p['dp']}_tp{p['tp']}_pp{p['pp']}"
    loss_s  = f"{p['final_loss']:.4f}" if p.get('final_loss') else "N/A"
    pow_s   = f"{p['observed_power_w']:.0f}" if p.get('observed_power_w') else "—"
    print(f"  {p['phase']:<6} {cfg_str:<22} {p['num_steps']:>8} "
          f"{p['train_s']:>8.1f}s {p['avg_iter_ms']:>9.0f}ms/it "
          f"{pow_s:>8} {loss_s:>10}")

In [ ]:
# ci check
CARBON_CSV = "/bgfs/stephenlee/wat29/scheduling-carbon-aware-3d-parallelism/data/carbon_synthetic_dataset.csv"
ci_csv     = pd.read_csv(CARBON_CSV)["carbon"].to_numpy()
n_buckets  = len(ci_csv)


In [ ]:
# two reference run
SMOKE_S    = 180.0
SMOKE_BUCK = 3600.0    # what the smoke run used
LONG_S     = 14400.0   # 4-hour planned comparison run

def ci_staircase(wall_s_grid, bucket_seconds, ci_array):
    return np.array([
        ci_array[min(int(t // bucket_seconds), len(ci_array) - 1)]
        for t in wall_s_grid
    ])


In [ ]:
# figure 1 full
fig, ax = plt.subplots(figsize=(12, 4))
hours = np.arange(n_buckets)
ax.plot(hours, ci_csv, color="#FF6F00", linewidth=1.2)
ax.fill_between(hours, 0, ci_csv, alpha=0.15, color="#FF6F00")

# smoke run window (in csv row units, given bucket=3600)
smoke_buckets = SMOKE_S / SMOKE_BUCK
ax.axvspan(0, smoke_buckets, color="#E91E63", alpha=0.6, zorder=3)
ax.annotate(
    f"smoke run\n{SMOKE_S:.0f}s @ bucket={SMOKE_BUCK:.0f}\n→ {smoke_buckets:.4f} buckets seen",
    xy=(smoke_buckets, ci_csv[0]), xytext=(20, ci_csv.max() * 0.88),
    fontsize=9, ha="left", color="#E91E63",
    arrowprops=dict(arrowstyle="->", color="#E91E63", lw=1.5),
)

# long run window at default bucket=3600
long_default_buckets = LONG_S / 3600.0
ax.axvspan(0, long_default_buckets, color="#1976D2", alpha=0.15, zorder=2)
ax.annotate(
    f"long run @ bucket=3600\n{LONG_S/3600:.0f}h → {long_default_buckets:.0f} buckets seen",
    xy=(long_default_buckets, ci_csv[int(long_default_buckets)]),
    xytext=(long_default_buckets + 60, ci_csv.max() * 0.55),
    fontsize=9, color="#1976D2",
    arrowprops=dict(arrowstyle="->", color="#1976D2", lw=1.2),
)

ax.set_xlabel("CSV row (= bucket index)", fontsize=10)
ax.set_ylabel("CI (gCO₂/kWh)", fontsize=10)
ax.set_title(f"Full CI Trace — {n_buckets} buckets ({n_buckets/24:.1f} days at 1h/bucket)",
             fontsize=11, fontweight="bold")
ax.text(0.02, 0.95,
        f"min={ci_csv.min():.0f}   max={ci_csv.max():.0f}   "
        f"mean={ci_csv.mean():.0f}   "
        f"max/min = {ci_csv.max()/ci_csv.min():.1f}×",
        transform=ax.transAxes, fontsize=9, va="top",
        bbox=dict(boxstyle="round", facecolor="white", edgecolor="#FF6F00"))
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "graph_ci_csv_full.png"),
            dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# figure 2 what
bucket_options = [3600, 1200, 600, 300, 60]
labels = {3600: "1 hour", 1200: "20 min", 600: "10 min",
          300: "5 min", 60: "1 min"}

fig, axes = plt.subplots(len(bucket_options), 1,
                          figsize=(11, 1.8 * len(bucket_options)),
                          sharex=True)
t_grid = np.arange(0, LONG_S, 1.0)   # 1-second resolution

for ax, b in zip(axes, bucket_options):
    seen = ci_staircase(t_grid, b, ci_csv)
    n_distinct = min(int(LONG_S / b) + 1, n_buckets)
    ax.step(t_grid, seen, where="post", color="#FF6F00", linewidth=1.4)
    ax.fill_between(t_grid, 0, seen, alpha=0.15,
                    color="#FF6F00", step="post")
    ax.set_title(
        f"bucket_seconds={b}  ({labels[b]} per CSV row)  "
        f"→ scheduler traverses {n_distinct} buckets in {LONG_S/3600:.0f}h, "
        f"CI range seen: {seen.min():.0f}–{seen.max():.0f} g/kWh",
        fontsize=9, fontweight="bold",
    )
    ax.set_ylabel("CI", fontsize=8)
    ax.tick_params(labelsize=8)

axes[-1].set_xlabel("Wall time (s) — full 4h run window", fontsize=10)
plt.suptitle("Scheduler's CI view at different bucket_seconds settings "
             "(LONG_S=14400s = 4h)",
             fontsize=11, fontweight="bold", y=1.00)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "graph_ci_staircase_options.png"),
            dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# summary
print("=" * 78)
print("CI TRACE ANALYSIS")
print("=" * 78)
print(f"  CSV:                {CARBON_CSV.split('/')[-1]}")
print(f"  Buckets:            {n_buckets} rows  ({n_buckets/24:.1f} days at 1h/bucket)")
print(f"  CI range:           {ci_csv.min():.0f} – {ci_csv.max():.0f} g/kWh "
      f"(max/min = {ci_csv.max()/ci_csv.min():.2f}×)")
print()
print(f"  Smoke run experience ({SMOKE_S:.0f}s @ bucket={SMOKE_BUCK:.0f}):")
print(f"    Saw {smoke_buckets:.4f} buckets → flat CI ≈ {ci_csv[0]:.0f}")
print(f"    Switch decision was driven by E_strat alone, not CI variation.")
print()
print(f"  Planned 4h comparison run:")
for b in bucket_options:
    n_seen = min(int(LONG_S / b) + 1, n_buckets)
    seen   = ci_csv[:n_seen]
    print(f"    bucket={b:>5} ({labels[b]:>8}):  "
          f"{n_seen:>3} buckets, CI {seen.min():.0f}–{seen.max():.0f} g/kWh, "
          f"max/min = {seen.max()/seen.min():.2f}×")
print()
print("  RECOMMENDATION for the long run:")
print("    --bucket-seconds 600   (10-min buckets → 24 buckets in 4h = 1 day of CI)")
print("    Decisions every ~200s (decide_every=1000 × τ≈0.2s) line up with")
print("    bucket changes every 600s — scheduler reacts to CI ~3× per bucket.")
print("=" * 78)

In [ ]:
# short carbon viz
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors

plt.rcParams.update({
    "font.family": "sans-serif", "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.3,
})


In [ ]:
# settings
BASE = os.getcwd()
MODEL_NAME = "gpt2_medium"
CALIB_STEM = "calibration_gpt2_medium_4gpu"
CARBON_CSV = os.path.join(BASE, "data", "carbon_short.csv")

# time compression knobs
BUCKET_SECONDS = 60          # wall seconds per CI row in the experiment
REAL_BUCKET_SECONDS = 3600   # what each CI row would represent in production
                             # (1 row = 1 hour is standard for grid ci forecasts)
COMPRESSION = REAL_BUCKET_SECONDS / BUCKET_SECONDS   # e.g. 60×

INTERVAL_SECONDS = 0.5
TRAIN_ITERS = 750
GLOBAL_BATCH = 16
SEQ_LEN = 1024
CI_FIXED = 400

DATA_DIR = os.path.join(BASE, "power_logs", MODEL_NAME)
PLOTS_DIR = os.path.join(BASE, "plots", MODEL_NAME)
os.makedirs(PLOTS_DIR, exist_ok=True)

print(f"Reading power logs from: {DATA_DIR}")
print(f"Reading CI CSV from:     {CARBON_CSV}")
print(f"Saving plots to:         {PLOTS_DIR}")
print(f"Time compression: {COMPRESSION:.0f}×  "
      f"({BUCKET_SECONDS}s experiment ↔ {REAL_BUCKET_SECONDS}s real)")

COLORS = {"dp": "#2196F3", "tp": "#4CAF50", "pp": "#FF9800",
          "mixed": "#9C27B0", "schedule": "#E91E63"}


In [ ]:
# load ci trace
ci_df = pd.read_csv(CARBON_CSV)
CI_VALUES = ci_df["carbon"].to_numpy(dtype=float)
SCHEDULE_TOTAL_S = len(CI_VALUES) * BUCKET_SECONDS
print(f"\nCI trace: {len(CI_VALUES)} buckets × {BUCKET_SECONDS}s = "
      f"{SCHEDULE_TOTAL_S}s experiment "
      f"(≈ {SCHEDULE_TOTAL_S * COMPRESSION / 3600:.1f} h real-world)")
print(f"  values: {CI_VALUES.tolist()}")

def ci_at_wall_time(n_samples, interval=INTERVAL_SECONDS):
    out = np.empty(n_samples, dtype=float)
    for i in range(n_samples):
        bucket = int((i * interval) // BUCKET_SECONDS)
        out[i] = CI_VALUES[min(bucket, len(CI_VALUES) - 1)]
    return out


In [ ]:
# ci bucket coloring
_CI_CMAP = plt.cm.YlOrRd
_uniq = sorted(set(CI_VALUES.tolist()))
_lo, _hi = (min(_uniq), max(_uniq)) if len(_uniq) > 1 else (_uniq[0]*0.9, _uniq[0]*1.1)
_CI_NORM = mcolors.Normalize(vmin=_lo, vmax=_hi)
def ci_to_color(c): return _CI_CMAP(_CI_NORM(c))

def annotate_ci_buckets(ax, alpha=0.18):
    for i, c in enumerate(CI_VALUES):
        x0, x1 = i * BUCKET_SECONDS, (i + 1) * BUCKET_SECONDS
        ax.axvspan(x0, x1, alpha=alpha, color=ci_to_color(c), zorder=0)

def add_ci_colorbar(fig, ax, label="CI (gCO₂/kWh)"):
    sm = plt.cm.ScalarMappable(cmap=_CI_CMAP, norm=_CI_NORM); sm.set_array([])
    cb = fig.colorbar(sm, ax=ax, orientation="vertical",
                      fraction=0.03, pad=0.02, shrink=0.75)
    cb.set_label(label, fontsize=8)
    cb.set_ticks(_uniq); cb.set_ticklabels([f"{c:g}" for c in _uniq])
    return cb

def add_real_time_axis(ax):
    """Top x-axis showing equivalent real-world time in hours.

    secondary_xaxis takes a (forward, inverse) pair of transforms that map
    bottom-axis units (experiment seconds) to top-axis units (real hours).
    matplotlib calls them whenever the bottom axis limits change, so the
    two axes stay in lockstep automatically.
    """
    secax = ax.secondary_xaxis(
        'top',
        functions=(lambda x: x * COMPRESSION / 3600.0,    # exp s → real h
                   lambda x: x * 3600.0 / COMPRESSION),   # real h → exp s
    )
    secax.set_xlabel("Equivalent real-world time (hours)", fontsize=8)
    secax.tick_params(labelsize=7)
    return secax


In [ ]:
# power csv loader
def load_power_csv(path):
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    if "time_step" not in df.columns:
        df.columns = ["timestamp", "total_power_W",
                      "GPU_0_W", "GPU_1_W", "GPU_2_W", "GPU_3_W"]
        df["time_step"] = 0
    return df

def compute_carbon(power_W, ci_arr, interval=INTERVAL_SECONDS):
    energy_kwh = power_W * interval / 3_600_000
    grams = energy_kwh * ci_arr
    return np.cumsum(grams), grams.sum()

def _classify(cfg):
    if cfg["is_schedule"]: return "schedule"
    dims_gt1 = sum(1 for d in (cfg["tp"], cfg["pp"], cfg["dp"]) if d > 1)
    if dims_gt1 >= 2: return "mixed"
    if cfg["tp"] > 1: return "tp"
    if cfg["pp"] > 1: return "pp"
    return "dp"

def _label(cfg):
    if cfg["is_schedule"]: return "Hybrid Schedule"
    parts = []
    if cfg["tp"] > 1: parts.append(f"TP={cfg['tp']}")
    if cfg["pp"] > 1: parts.append(f"PP={cfg['pp']}")
    if cfg["dp"] > 1: parts.append(f"DP={cfg['dp']}")
    return ",".join(parts) if parts else "DP=1"


In [ ]:
# discover load runs
all_static = {
    "tp1_pp1_dp4": {"tp": 1, "pp": 1, "dp": 4, "is_schedule": False},
    "tp4_pp1_dp1": {"tp": 4, "pp": 1, "dp": 1, "is_schedule": False},
    "tp1_pp4_dp1": {"tp": 1, "pp": 4, "dp": 1, "is_schedule": False},
    "tp2_pp2_dp1": {"tp": 2, "pp": 2, "dp": 1, "is_schedule": False},
    "tp2_pp1_dp2": {"tp": 2, "pp": 1, "dp": 2, "is_schedule": False},
    "tp1_pp2_dp2": {"tp": 1, "pp": 2, "dp": 2, "is_schedule": False},
}
for k, v in all_static.items():
    v["file"] = f"elastic_{k}_power.csv"

hybrid_def = {
    "hybrid_schedule": {
        "tp": None, "pp": None, "dp": None,
        "file": f"online_elastic_{CALIB_STEM}_power.csv",
        "is_schedule": True,
    }
}

configs, skipped = {}, []
for name, cfg in {**all_static, **hybrid_def}.items():
    path = os.path.join(DATA_DIR, cfg["file"])
    if not os.path.exists(path) or os.path.getsize(path) < 100:
        skipped.append(name); continue
    df = load_power_csv(path)
    if len(df) < 2:
        skipped.append(name); continue
    warmup = int(len(df) * 0.2)
    cfg["avg_power"] = df["total_power_W"].mean()
    cfg["steady_avg_power"] = df.iloc[warmup:]["total_power_W"].mean()
    cfg["peak_power"] = df["total_power_W"].max()
    cfg["df"] = df
    configs[name] = cfg
    tag = " [SCHED]" if cfg["is_schedule"] else ""
    print(f"  {name}{tag}: avg={cfg['avg_power']:.0f}W "
          f"steady={cfg['steady_avg_power']:.0f}W "
          f"samples={len(df)} ({len(df)*INTERVAL_SECONDS:.0f}s)")
if skipped:
    print(f"  skipped (no data): {', '.join(skipped)}")
if not configs:
    raise RuntimeError(f"No power CSVs found in {DATA_DIR}")

all_configs = [(_label(cfg), _classify(cfg), cfg) for _, cfg in configs.items()]


In [ ]:
# metrics
total_tokens = TRAIN_ITERS * GLOBAL_BATCH * SEQ_LEN
print(f"\n{'Label':22s} | {'Time':>6s} | {'Power':>6s} | "
      f"{'Energy':>7s} | {'C_fix':>6s} | {'C_var':>6s} | {'Tput':>9s}")
print("-" * 82)
for label, _, cfg in all_configs:
    P = cfg["df"]["total_power_W"].to_numpy()
    n = len(P)
    cfg["train_time_s"] = n * INTERVAL_SECONDS
    cfg["avg_power_w"] = P.mean()
    cfg["total_energy_wh"] = P.sum() * INTERVAL_SECONDS / 3600
    cfg["tokens_per_sec"] = total_tokens / cfg["train_time_s"]
    _, cfg["carbon_fixed_g"] = compute_carbon(P, np.full(n, CI_FIXED))
    _, cfg["carbon_varying_g"] = compute_carbon(P, ci_at_wall_time(n))
    tag = " ★" if cfg["is_schedule"] else ""
    print(f"{label.replace(chr(10),' '):22s} | "
          f"{cfg['train_time_s']:5.1f}s | "
          f"{cfg['avg_power_w']:5.0f}W | "
          f"{cfg['total_energy_wh']:6.2f}Wh | "
          f"{cfg['carbon_fixed_g']:5.1f}g | "
          f"{cfg['carbon_varying_g']:5.1f}g | "
          f"{cfg['tokens_per_sec']:7.1f} t/s{tag}")


In [ ]:
# cumulative carbon dual
fig, axes = plt.subplots(1, 2, figsize=(14, 5.4))
for ax, use_var in zip(axes, [False, True]):
    for name, cfg in configs.items():
        P = cfg["df"]["total_power_W"].to_numpy()
        n = len(P)
        ci = ci_at_wall_time(n) if use_var else np.full(n, CI_FIXED)
        cum, _ = compute_carbon(P, ci)
        x = np.arange(n) * INTERVAL_SECONDS
        kw = {"linewidth": 2.4, "linestyle": "--"} if cfg["is_schedule"] \
             else {"linewidth": 1.7}
        col = COLORS["schedule"] if cfg["is_schedule"] else None
        lbl = "Hybrid Schedule" if cfg["is_schedule"] else \
              f"TP={cfg['tp']}, PP={cfg['pp']}, DP={cfg['dp']}"
        ax.plot(x, cum, label=lbl, color=col, **kw)
        ax.scatter(x[-1], cum[-1], s=42, zorder=3, color=col)

    if use_var:
        annotate_ci_buckets(ax)
        ax.set_title(f"Varying CI  (from {os.path.basename(CARBON_CSV)})",
                     fontsize=10, fontweight="bold")
        add_ci_colorbar(fig, ax)
    else:
        ax.set_title(f"Fixed CI  ({CI_FIXED} gCO₂/kWh)",
                     fontsize=10, fontweight="bold")

    ax.set_xlabel("Experiment time (s)", fontsize=9)
    ax.set_ylabel("Cumulative CO₂ (g)", fontsize=9)
    ax.legend(fontsize=8, ncol=2, frameon=False, loc="upper left")
    add_real_time_axis(ax)

plt.suptitle(
    f"Cumulative Carbon Emissions — Static Baselines vs. Hybrid\n"
    f"({COMPRESSION:.0f}× time compression: 1s experiment ≈ {COMPRESSION:.0f}s real-world)",
    fontsize=11, fontweight="bold", y=1.02
)
plt.tight_layout()
out = os.path.join(PLOTS_DIR, "graph_carbon_varying_ci.png")
plt.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
print(f"\nSaved: {out}")


In [ ]:
# power traces config
n_cfg = len(configs)
ncols = min(3, n_cfg); nrows = (n_cfg + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(13, 3.0 * nrows))
axes = np.atleast_1d(axes).flatten()

for ax, (name, cfg) in zip(axes, configs.items()):
    df = cfg["df"]
    x = np.arange(len(df)) * INTERVAL_SECONDS
    y = df["total_power_W"].to_numpy()
    warm = int(len(df) * 0.2)
    color = COLORS["schedule"] if cfg["is_schedule"] else "#1976D2"
    ax.plot(x[:warm], y[:warm], color="#B0BEC5", lw=1.4)
    ax.plot(x[warm:], y[warm:], color=color, lw=1.6)
    ax.axhline(cfg["steady_avg_power"], color="#43A047", ls=":", lw=1)

    if cfg["is_schedule"]:
        steps = df["time_step"].to_numpy()
        for j in range(1, len(steps)):
            if steps[j] < steps[j-1] and steps[j-1] > 0:
                ax.axvline(x[j], color="#FF9800", ls="-.", lw=0.8, alpha=0.7)
        title = "Hybrid Schedule"
    else:
        title = f"TP={cfg['tp']}, PP={cfg['pp']}, DP={cfg['dp']}"
    ax.set_title(title, fontsize=9, fontweight="bold")
    ax.set_xlabel("Experiment time (s)", fontsize=8)
    ax.set_ylabel("Power (W)", fontsize=8)
    ax.tick_params(labelsize=7)
    ax.text(0.02, 0.92, f"avg={cfg['steady_avg_power']:.0f}W",
            transform=ax.transAxes, fontsize=7, color="#2E7D32")
    add_real_time_axis(ax)

for ax in axes[n_cfg:]:
    ax.set_visible(False)

fig.suptitle(
    f"Power Per Configuration  "
    f"(top axis: real-world equivalent at {COMPRESSION:.0f}× compression)",
    fontsize=11, fontweight="bold", y=1.01
)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "graph0_power_trends.png"),
            dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# bar grid no
labels_bar = [c[0] for c in all_configs]
colors_bar = [COLORS[c[1]] for c in all_configs]
edge       = ["#E91E63" if c[2]["is_schedule"] else "white" for c in all_configs]
ew         = [2.5      if c[2]["is_schedule"] else 0.5     for c in all_configs]
time_v   = [c[2]["train_time_s"]    for c in all_configs]
energy_v = [c[2]["total_energy_wh"] for c in all_configs]
c_fix    = [c[2]["carbon_fixed_g"]  for c in all_configs]
c_var    = [c[2]["carbon_varying_g"]for c in all_configs]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

ax = axes[0]
ax.bar(labels_bar, time_v, color=colors_bar, edgecolor=edge, linewidth=ew)
for i, v in enumerate(time_v):
    ax.text(i, v + max(time_v)*0.01,
            f"{v:.0f}s\n(~{v*COMPRESSION/3600:.1f}h real)",
            ha="center", va="bottom", fontsize=6.5)
ax.set_ylabel("Experiment time (s)")
ax.set_title("Total Time", fontweight="bold")
ax.tick_params(axis="x", labelsize=7, rotation=20)

ax = axes[1]
ax.bar(labels_bar, energy_v, color=colors_bar, edgecolor=edge, linewidth=ew)
for i, v in enumerate(energy_v):
    ax.text(i, v + max(energy_v)*0.01, f"{v:.1f}",
            ha="center", va="bottom", fontsize=7)
ax.set_ylabel("Energy (Wh)")
ax.set_title("Total Energy", fontweight="bold")
ax.tick_params(axis="x", labelsize=7, rotation=20)

ax = axes[2]
xpos = np.arange(len(labels_bar)); w = 0.38
ax.bar(xpos - w/2, c_fix, w, color=colors_bar, alpha=0.55,
       edgecolor=edge, linewidth=ew, label=f"Fixed CI ({CI_FIXED})")
ax.bar(xpos + w/2, c_var, w, color=colors_bar,
       edgecolor="black", linewidth=0.6, label="Varying CI (CSV)")
for xi, vv in zip(xpos, c_var):
    ax.text(xi + w/2, vv + max(c_var)*0.01, f"{vv:.0f}",
            ha="center", va="bottom", fontsize=6.5)
ax.set_xticks(xpos); ax.set_xticklabels(labels_bar, fontsize=7, rotation=20)
ax.set_ylabel("Carbon (gCO₂)")
ax.set_title("Carbon — Fixed vs Varying CI", fontweight="bold")
ax.legend(fontsize=7, loc="upper left")

patches = [
    mpatches.Patch(color=COLORS["dp"],       label="Data Parallel"),
    mpatches.Patch(color=COLORS["tp"],       label="Tensor Parallel"),
    mpatches.Patch(color=COLORS["pp"],       label="Pipeline Parallel"),
    mpatches.Patch(color=COLORS["mixed"],    label="Mixed"),
    mpatches.Patch(color=COLORS["schedule"], label="Hybrid Schedule"),
]
fig.legend(handles=patches, fontsize=8, ncol=5,
           loc="upper center", bbox_to_anchor=(0.5, 1.03), frameon=False)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "graph_metrics_overview.png"),
            dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# summary
rows = [{
    "Config":           c[0],
    "Type":             "Schedule" if c[2]["is_schedule"] else "Static",
    "ExpTime(s)":       round(c[2]["train_time_s"], 1),
    "RealTime(h)":      round(c[2]["train_time_s"] * COMPRESSION / 3600, 2),
    "AvgPower(W)":      round(c[2]["avg_power_w"], 1),
    "Energy(Wh)":       round(c[2]["total_energy_wh"], 2),
    f"C_Fixed@{CI_FIXED}(g)": round(c[2]["carbon_fixed_g"], 2),
    "C_Varying(g)":     round(c[2]["carbon_varying_g"], 2),
    "Tput(tok/s)":      round(c[2]["tokens_per_sec"], 1),
} for c in all_configs]
summary_df = pd.DataFrame(rows)
print("\n" + "=" * 100)
print("SUMMARY")
print("=" * 100)
print(summary_df.to_string(index=False))
out_csv = os.path.join(PLOTS_DIR, "summary_table.csv")
summary_df.to_csv(out_csv, index=False)
print(f"\nSaved: {out_csv}")

# verdict
if any(c[2]["is_schedule"] for c in all_configs):
    hybrid_var = next(c[2]["carbon_varying_g"]
                      for c in all_configs if c[2]["is_schedule"])
    static_var = [c[2]["carbon_varying_g"]
                  for c in all_configs if not c[2]["is_schedule"]]
    if static_var:
        best_static = min(static_var)
        savings = (best_static - hybrid_var) / best_static * 100
        verdict = "WINS" if hybrid_var < best_static else "LOSES TO BEST STATIC"
        print(f"\nHybrid: {hybrid_var:.2f} g    "
              f"Best static: {best_static:.2f} g    "
              f"Hybrid {verdict} ({savings:+.1f}%)")

In [ ]:
# visualize long run
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors

plt.rcParams.update({
    "font.family": "sans-serif", "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.3,
})


In [ ]:
# settings
BASE         = os.getcwd()
MODEL_NAME   = "gpt2_medium"
CALIB_STEM   = "calibration_gpt2_medium_4gpu"
CARBON_CSV   = os.path.join(BASE, "data", "carbon_synthetic_dataset.csv")
TRAIN_ITERS  = 20000          # match your sbatch's TRAIN_ITERS

BUCKET_SECONDS      = 60      # match sbatch's BUCKET_SEC
REAL_BUCKET_SECONDS = 3600
COMPRESSION         = REAL_BUCKET_SECONDS / BUCKET_SECONDS

INTERVAL_SECONDS = 0.5
GLOBAL_BATCH     = 16
SEQ_LEN          = 1024
CI_FIXED         = 400        # baseline-comparison CI for the "fixed CI" panel

# them intact only sustained tails get cut.
IDLE_W      = 250
MIN_IDLE_S  = 5

DATA_DIR  = os.path.join(BASE, "power_logs", MODEL_NAME)
PLOTS_DIR = os.path.join(BASE, "plots", MODEL_NAME + "_long")
os.makedirs(PLOTS_DIR, exist_ok=True)

print(f"Power logs:  {DATA_DIR}")
print(f"CI CSV:      {CARBON_CSV}")
print(f"Compression: {COMPRESSION:.0f}× (1 experiment-s ≈ {COMPRESSION:.0f}s real)")

COLORS = {"dp": "#2196F3", "tp": "#4CAF50", "pp": "#FF9800",
          "mixed": "#9C27B0", "schedule": "#E91E63"}


In [ ]:
# load ci trace
CI_VALUES = pd.read_csv(CARBON_CSV)["carbon"].to_numpy(dtype=float)
print(f"CI trace: {len(CI_VALUES)} buckets × {BUCKET_SECONDS}s "
      f"= {len(CI_VALUES) * BUCKET_SECONDS}s horizon")

def ci_at_wall(n_samples, interval=INTERVAL_SECONDS):
    out = np.empty(n_samples)
    for i in range(n_samples):
        b = int((i * interval) // BUCKET_SECONDS)
        out[i] = CI_VALUES[min(b, len(CI_VALUES) - 1)]
    return out


In [ ]:
# ci colors
_CI_CMAP = plt.cm.YlOrRd
_lo, _hi = CI_VALUES.min(), CI_VALUES.max()
_CI_NORM = mcolors.Normalize(vmin=_lo, vmax=_hi)

def annotate_ci_buckets(ax, alpha=0.18):
    for i, c in enumerate(CI_VALUES):
        x0, x1 = i * BUCKET_SECONDS, (i + 1) * BUCKET_SECONDS
        ax.axvspan(x0, x1, alpha=alpha, color=_CI_CMAP(_CI_NORM(c)), zorder=0)

def add_ci_colorbar(fig, ax, label="CI (gCO₂/kWh)"):
    sm = plt.cm.ScalarMappable(cmap=_CI_CMAP, norm=_CI_NORM); sm.set_array([])
    cb = fig.colorbar(sm, ax=ax, fraction=0.03, pad=0.02, shrink=0.75)
    cb.set_label(label, fontsize=8)

def add_real_time_axis(ax):
    secax = ax.secondary_xaxis(
        'top',
        functions=(lambda x: x * COMPRESSION / 3600.0,
                   lambda x: x * 3600.0 / COMPRESSION),
    )
    secax.set_xlabel("Equivalent real-world time (hours)", fontsize=8)
    secax.tick_params(labelsize=7)


In [ ]:
# load runs
def load_power(path):
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    if "time_step" not in df.columns:
        df.columns = ["timestamp", "total_power_W",
                      "GPU_0_W", "GPU_1_W", "GPU_2_W", "GPU_3_W"]
        df["time_step"] = 0
    return df

def trim_idle_tail(df, training_w=500):
    """Cut everything after the last training-level power reading.

    Watchdog tails sit at ~280W with brief rebumps to ~330W as Megatron
    prints its summary and the signal handler runs. Anything below
    `training_w` is post-training cleanup, regardless of whether the
    file happens to end during a rebump or during the idle plateau.
    """
    P = df["total_power_W"].to_numpy()
    if len(P) < 2:
        return df, 0
    above = np.where(P >= training_w)[0]
    if len(above) == 0:
        return df, 0          # never reached training level
    last_train = above[-1]
    cut = last_train + 1
    if cut >= len(P):
        return df, 0          # already ends at training level
    return df.iloc[:cut].reset_index(drop=True), len(P) - cut

def carbon(power_W, ci_arr, interval=INTERVAL_SECONDS):
    grams = power_W * interval / 3_600_000 * ci_arr
    return np.cumsum(grams), grams.sum()

run_specs = {
    "tp1_pp1_dp4":     {"tp": 1, "pp": 1, "dp": 4, "is_schedule": False,
                        "file": "elastic_tp1_pp1_dp4_power.csv"},
    "tp2_pp1_dp2":     {"tp": 2, "pp": 1, "dp": 2, "is_schedule": False,
                        "file": "elastic_tp2_pp1_dp2_power.csv"},
    "hybrid_schedule": {"tp": None, "pp": None, "dp": None, "is_schedule": True,
                        "file": f"online_elastic_{CALIB_STEM}_power.csv"},
}

configs, skipped = {}, []
for name, cfg in run_specs.items():
    path = os.path.join(DATA_DIR, cfg["file"])
    if not os.path.exists(path) or os.path.getsize(path) < 100:
        skipped.append(name); continue
    df = load_power(path)
    df, dropped = trim_idle_tail(df)
    if len(df) < 2:
        skipped.append(name); continue
    cfg["df"] = df
    configs[name] = cfg
    tag = " [SCHED]" if cfg["is_schedule"] else ""
    tail_msg = f", trimmed {dropped*INTERVAL_SECONDS:.0f}s idle tail" if dropped else ""
    print(f"  loaded {name}{tag}: {len(df)} samples, "
          f"{len(df) * INTERVAL_SECONDS:.0f}s{tail_msg}")
if skipped:
    print(f"  skipped: {', '.join(skipped)}")
assert configs, f"no power CSVs found in {DATA_DIR}"

def label(cfg):
    if cfg["is_schedule"]: return "Hybrid Schedule"
    parts = []
    if cfg["tp"] > 1: parts.append(f"TP={cfg['tp']}")
    if cfg["pp"] > 1: parts.append(f"PP={cfg['pp']}")
    if cfg["dp"] > 1: parts.append(f"DP={cfg['dp']}")
    return ",".join(parts) or "DP=1"

def kind(cfg):
    if cfg["is_schedule"]: return "schedule"
    if sum(1 for d in (cfg["tp"], cfg["pp"], cfg["dp"]) if d > 1) >= 2: return "mixed"
    if cfg["pp"] > 1: return "pp"
    if cfg["tp"] > 1: return "tp"
    return "dp"


In [ ]:
# metrics
total_tokens = TRAIN_ITERS * GLOBAL_BATCH * SEQ_LEN
rows = []
for name, cfg in configs.items():
    P = cfg["df"]["total_power_W"].to_numpy()
    n = len(P)
    train_s = n * INTERVAL_SECONDS
    avg_p   = P.mean()
    energy  = P.sum() * INTERVAL_SECONDS / 3600
    _, c_fix = carbon(P, np.full(n, CI_FIXED))
    _, c_var = carbon(P, ci_at_wall(n))
    rows.append({
        "Config":               label(cfg),
        "Type":                 "Schedule" if cfg["is_schedule"] else "Static",
        "ExpTime(s)":           round(train_s, 1),
        "RealTime(h)":          round(train_s * COMPRESSION / 3600, 2),
        "AvgPower(W)":          round(avg_p, 1),
        "Energy(Wh)":           round(energy, 2),
        f"C_Fixed@{CI_FIXED}(g)": round(c_fix, 2),
        "C_Varying(g)":         round(c_var, 2),
        "Tput(tok/s)":          round(total_tokens / train_s, 1),
    })
    cfg.update(train_time_s=train_s, avg_power_w=avg_p,
               total_energy_wh=energy, carbon_fixed_g=c_fix,
               carbon_varying_g=c_var)

summary_df = pd.DataFrame(rows)
print("\n" + "=" * 100)
print(summary_df.to_string(index=False))
print("=" * 100)
summary_df.to_csv(os.path.join(PLOTS_DIR, "summary_table.csv"), index=False)


In [ ]:
# reshard overhead
# the overhead separately for transparency.
print("\nReshard overhead (mid-run idle, post tail-trim):")
for name, cfg in configs.items():
    if not cfg["is_schedule"]:
        continue
    P = cfg["df"]["total_power_W"].to_numpy()
    ci_arr = ci_at_wall(len(P))
    warm = int(len(P) * 0.1)
    idle_mask = np.zeros(len(P), dtype=bool)
    idle_mask[warm:] = P[warm:] < IDLE_W
    n_idle  = idle_mask.sum()
    idle_s  = n_idle * INTERVAL_SECONDS
    idle_g = (P[idle_mask] * INTERVAL_SECONDS / 3_600_000
              * ci_arr[idle_mask]).sum()
    pct = idle_g / cfg["carbon_varying_g"] * 100 if cfg["carbon_varying_g"] else 0
    print(f"  {label(cfg):20s}: ~{idle_s:.0f}s idle, "
          f"~{idle_g:.2f}g ({pct:.1f}% of total carbon)")

# quick result
schedules = [c for _, c in configs.items() if c["is_schedule"]]
statics   = [c for _, c in configs.items() if not c["is_schedule"]]
if schedules and statics:
    h_var = schedules[0]["carbon_varying_g"]
    s_var = min(c["carbon_varying_g"] for c in statics)
    pct = (s_var - h_var) / s_var * 100
    print(f"\nHybrid: {h_var:.2f}g    Best static: {s_var:.2f}g    "
          f"Hybrid {'WINS' if h_var < s_var else 'LOSES TO BEST STATIC'} "
          f"({pct:+.1f}%)")


In [ ]:
# plot power
fig, axes = plt.subplots(1, 2, figsize=(14, 5.4))
for ax, varying in zip(axes, [False, True]):
    for _, cfg in configs.items():
        P = cfg["df"]["total_power_W"].to_numpy()
        n = len(P)
        ci = ci_at_wall(n) if varying else np.full(n, CI_FIXED)
        cum, _ = carbon(P, ci)
        x = np.arange(n) * INTERVAL_SECONDS
        kw = {"linewidth": 2.4, "linestyle": "--"} if cfg["is_schedule"] else {"linewidth": 1.7}
        col = COLORS["schedule"] if cfg["is_schedule"] else None
        ax.plot(x, cum, label=label(cfg), color=col, **kw)
        ax.scatter(x[-1], cum[-1], s=42, zorder=3, color=col)
    if varying:
        annotate_ci_buckets(ax)
        ax.set_title(f"Varying CI (from {os.path.basename(CARBON_CSV)})",
                     fontsize=10, fontweight="bold")
        add_ci_colorbar(fig, ax)
    else:
        ax.set_title(f"Fixed CI ({CI_FIXED} gCO₂/kWh)",
                     fontsize=10, fontweight="bold")
    ax.set_xlabel("Experiment time (s)")
    ax.set_ylabel("Cumulative CO₂ (g)")
    ax.legend(fontsize=9, frameon=False, loc="upper left")
    add_real_time_axis(ax)
plt.suptitle(
    f"Long-Run Carbon: {TRAIN_ITERS} iters, "
    f"{COMPRESSION:.0f}× time compression",
    fontsize=11, fontweight="bold", y=1.02
)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "graph_carbon_long.png"),
            dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# plot energy
n_cfg = len(configs)
ncols = min(3, n_cfg); nrows = (n_cfg + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(13, 3.0 * nrows))
axes = np.atleast_1d(axes).flatten()

for ax, (_, cfg) in zip(axes, configs.items()):
    df = cfg["df"]
    x = np.arange(len(df)) * INTERVAL_SECONDS
    y = df["total_power_W"].to_numpy()
    warm = int(len(df) * 0.2)
    color = COLORS["schedule"] if cfg["is_schedule"] else "#1976D2"
    ax.plot(x[:warm], y[:warm], color="#B0BEC5", lw=1.4)
    ax.plot(x[warm:], y[warm:], color=color, lw=1.6)
    steady = df["total_power_W"].iloc[warm:].mean()
    ax.axhline(steady, color="#43A047", ls=":", lw=1)

    if cfg["is_schedule"]:
        steps = df["time_step"].to_numpy()
        for j in range(1, len(steps)):
            if steps[j] < steps[j-1] and steps[j-1] > 0:
                ax.axvline(x[j], color="#FF9800", ls="-.", lw=0.8, alpha=0.7)
        title = "Hybrid Schedule"
    else:
        title = f"TP={cfg['tp']}, PP={cfg['pp']}, DP={cfg['dp']}"
    ax.set_title(title, fontsize=9, fontweight="bold")
    ax.set_xlabel("Experiment time (s)", fontsize=8)
    ax.set_ylabel("Power (W)", fontsize=8)
    ax.tick_params(labelsize=7)
    ax.text(0.02, 0.92, f"avg={steady:.0f}W",
            transform=ax.transAxes, fontsize=7, color="#2E7D32")
    add_real_time_axis(ax)

for ax in axes[n_cfg:]:
    ax.set_visible(False)
fig.suptitle(f"Power Per Configuration "
             f"(top axis: real-world equivalent at {COMPRESSION:.0f}× compression)",
             fontsize=11, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "graph0_power_trends.png"),
            dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# plot 3 bar
ordered = list(configs.values())
labels  = [label(c) for c in ordered]
cols    = [COLORS[kind(c)] for c in ordered]
edges   = ["#E91E63" if c["is_schedule"] else "white" for c in ordered]
ews     = [2.5 if c["is_schedule"] else 0.5 for c in ordered]
times   = [c["train_time_s"]    for c in ordered]
energ   = [c["total_energy_wh"] for c in ordered]
cfix    = [c["carbon_fixed_g"]  for c in ordered]
cvar    = [c["carbon_varying_g"]for c in ordered]

fig, axs = plt.subplots(1, 3, figsize=(16, 4))

axs[0].bar(labels, times, color=cols, edgecolor=edges, linewidth=ews)
for i, v in enumerate(times):
    axs[0].text(i, v + max(times)*0.01,
                f"{v:.0f}s\n(~{v*COMPRESSION/3600:.1f}h real)",
                ha="center", va="bottom", fontsize=6.5)
axs[0].set_ylabel("Experiment time (s)")
axs[0].set_title("Total Time", fontweight="bold")
axs[0].tick_params(axis="x", labelsize=8, rotation=15)

axs[1].bar(labels, energ, color=cols, edgecolor=edges, linewidth=ews)
for i, v in enumerate(energ):
    axs[1].text(i, v + max(energ)*0.01, f"{v:.1f}",
                ha="center", va="bottom", fontsize=8)
axs[1].set_ylabel("Energy (Wh)")
axs[1].set_title("Total Energy", fontweight="bold")
axs[1].tick_params(axis="x", labelsize=8, rotation=15)

xpos = np.arange(len(labels)); w = 0.38
axs[2].bar(xpos - w/2, cfix, w, color=cols, alpha=0.55,
           edgecolor=edges, linewidth=ews, label=f"Fixed CI ({CI_FIXED})")
axs[2].bar(xpos + w/2, cvar, w, color=cols,
           edgecolor="black", linewidth=0.6, label="Varying CI")
for xi, v in zip(xpos, cvar):
    axs[2].text(xi + w/2, v + max(cvar)*0.01, f"{v:.0f}",
                ha="center", va="bottom", fontsize=7)
axs[2].set_xticks(xpos); axs[2].set_xticklabels(labels, fontsize=8, rotation=15)
axs[2].set_ylabel("Carbon (gCO₂)")
axs[2].set_title("Carbon — Fixed vs Varying CI", fontweight="bold")
axs[2].legend(fontsize=8, loc="upper left")

patches = [
    mpatches.Patch(color=COLORS["dp"],       label="Data Parallel"),
    mpatches.Patch(color=COLORS["mixed"],    label="Mixed (TP+DP)"),
    mpatches.Patch(color=COLORS["schedule"], label="Hybrid Schedule"),
]
fig.legend(handles=patches, fontsize=9, ncol=3,
           loc="upper center", bbox_to_anchor=(0.5, 1.03), frameon=False)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "graph_metrics_overview.png"),
            dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# visualize long run
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors

plt.rcParams.update({
    "font.family": "sans-serif", "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.3,
})


In [ ]:
# settings
BASE         = os.getcwd()
MODEL_NAME   = "gpt2_medium"
CALIB_STEM   = "calibration_gpt2_medium_4gpu"
CARBON_CSV   = os.path.join(BASE, "data", "carbon_synthetic_dataset.csv")
TRAIN_ITERS  = 20000          # match your sbatch's TRAIN_ITERS

BUCKET_SECONDS   = 60         # match sbatch's BUCKET_SEC — wall seconds per CI row
INTERVAL_SECONDS = 0.5        # PowerMonitor sampling interval
GLOBAL_BATCH     = 16
SEQ_LEN          = 1024
CI_FIXED         = 400        # baseline-comparison CI for the "fixed CI" panel

DATA_DIR  = os.path.join(BASE, "power_logs", MODEL_NAME)
PLOTS_DIR = os.path.join(BASE, "plots", MODEL_NAME + "_long")
os.makedirs(PLOTS_DIR, exist_ok=True)

print(f"Power logs:  {DATA_DIR}")
print(f"CI CSV:      {CARBON_CSV}")
print(f"CI bucket:   {BUCKET_SECONDS}s per row")

COLORS = {"dp": "#2196F3", "tp": "#4CAF50", "pp": "#FF9800",
          "mixed": "#9C27B0", "schedule": "#E91E63"}


In [ ]:
# load ci trace
CI_VALUES = pd.read_csv(CARBON_CSV)["carbon"].to_numpy(dtype=float)
print(f"CI trace: {len(CI_VALUES)} buckets × {BUCKET_SECONDS}s "
      f"= {len(CI_VALUES) * BUCKET_SECONDS}s horizon")

def ci_at_wall(n_samples, interval=INTERVAL_SECONDS):
    out = np.empty(n_samples)
    for i in range(n_samples):
        b = int((i * interval) // BUCKET_SECONDS)
        out[i] = CI_VALUES[min(b, len(CI_VALUES) - 1)]
    return out


In [ ]:
# ci colors
_CI_CMAP = plt.cm.YlOrRd
_lo, _hi = CI_VALUES.min(), CI_VALUES.max()
_CI_NORM = mcolors.Normalize(vmin=_lo, vmax=_hi)

def annotate_ci_buckets(ax, alpha=0.18):
    for i, c in enumerate(CI_VALUES):
        x0, x1 = i * BUCKET_SECONDS, (i + 1) * BUCKET_SECONDS
        ax.axvspan(x0, x1, alpha=alpha, color=_CI_CMAP(_CI_NORM(c)), zorder=0)

def add_ci_colorbar(fig, ax, label="CI (gCO₂/kWh)"):
    sm = plt.cm.ScalarMappable(cmap=_CI_CMAP, norm=_CI_NORM); sm.set_array([])
    cb = fig.colorbar(sm, ax=ax, fraction=0.03, pad=0.02, shrink=0.75)
    cb.set_label(label, fontsize=8)


In [ ]:
# load runs
def load_power(path):
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    if "time_step" not in df.columns:
        df.columns = ["timestamp", "total_power_W",
                      "GPU_0_W", "GPU_1_W", "GPU_2_W", "GPU_3_W"]
        df["time_step"] = 0
    return df

def trim_idle_tail(df, training_w=500):
    """Cut everything after the last training-level power reading.

    Watchdog tails sit at ~280W with brief rebumps to ~330W as Megatron
    prints its summary and the signal handler runs. Anything below
    `training_w` is post-training cleanup, regardless of whether the
    file happens to end during a rebump or during the idle plateau.
    """
    P = df["total_power_W"].to_numpy()
    if len(P) < 2:
        return df, 0
    above = np.where(P >= training_w)[0]
    if len(above) == 0:
        return df, 0          # never reached training level
    last_train = above[-1]
    cut = last_train + 1
    if cut >= len(P):
        return df, 0          # already ends at training level
    return df.iloc[:cut].reset_index(drop=True), len(P) - cut

def carbon(power_W, ci_arr, interval=INTERVAL_SECONDS):
    grams = power_W * interval / 3_600_000 * ci_arr
    return np.cumsum(grams), grams.sum()

run_specs = {
    "tp1_pp1_dp4":     {"tp": 1, "pp": 1, "dp": 4, "is_schedule": False,
                        "file": "elastic_tp1_pp1_dp4_power.csv"},
    "tp2_pp1_dp2":     {"tp": 2, "pp": 1, "dp": 2, "is_schedule": False,
                        "file": "elastic_tp2_pp1_dp2_power.csv"},
    "hybrid_schedule": {"tp": None, "pp": None, "dp": None, "is_schedule": True,
                        "file": f"online_elastic_{CALIB_STEM}_power.csv"},
}

configs, skipped = {}, []
for name, cfg in run_specs.items():
    path = os.path.join(DATA_DIR, cfg["file"])
    if not os.path.exists(path) or os.path.getsize(path) < 100:
        skipped.append(name); continue
    df = load_power(path)
    df, dropped = trim_idle_tail(df)
    if len(df) < 2:
        skipped.append(name); continue
    cfg["df"] = df
    configs[name] = cfg
    tag = " [SCHED]" if cfg["is_schedule"] else ""
    tail_msg = f", trimmed {dropped*INTERVAL_SECONDS:.0f}s idle tail" if dropped else ""
    print(f"  loaded {name}{tag}: {len(df)} samples, "
          f"{len(df) * INTERVAL_SECONDS:.0f}s{tail_msg}")
if skipped:
    print(f"  skipped: {', '.join(skipped)}")
assert configs, f"no power CSVs found in {DATA_DIR}"

def label(cfg):
    if cfg["is_schedule"]: return "Hybrid Schedule"
    parts = []
    if cfg["tp"] > 1: parts.append(f"TP={cfg['tp']}")
    if cfg["pp"] > 1: parts.append(f"PP={cfg['pp']}")
    if cfg["dp"] > 1: parts.append(f"DP={cfg['dp']}")
    return ",".join(parts) or "DP=1"

def kind(cfg):
    if cfg["is_schedule"]: return "schedule"
    if sum(1 for d in (cfg["tp"], cfg["pp"], cfg["dp"]) if d > 1) >= 2: return "mixed"
    if cfg["pp"] > 1: return "pp"
    if cfg["tp"] > 1: return "tp"
    return "dp"


In [ ]:
# metrics
total_tokens = TRAIN_ITERS * GLOBAL_BATCH * SEQ_LEN
rows = []
for name, cfg in configs.items():
    P = cfg["df"]["total_power_W"].to_numpy()
    n = len(P)
    train_s = n * INTERVAL_SECONDS
    avg_p   = P.mean()
    energy  = P.sum() * INTERVAL_SECONDS / 3600
    _, c_fix = carbon(P, np.full(n, CI_FIXED))
    _, c_var = carbon(P, ci_at_wall(n))
    rows.append({
        "Config":               label(cfg),
        "Type":                 "Schedule" if cfg["is_schedule"] else "Static",
        "Wall(s)":              round(train_s, 1),
        "AvgPower(W)":          round(avg_p, 1),
        "Energy(Wh)":           round(energy, 2),
        f"C_Fixed@{CI_FIXED}(g)": round(c_fix, 2),
        "C_Varying(g)":         round(c_var, 2),
        "Tput(tok/s)":          round(total_tokens / train_s, 1),
    })
    cfg.update(train_time_s=train_s, avg_power_w=avg_p,
               total_energy_wh=energy, carbon_fixed_g=c_fix,
               carbon_varying_g=c_var)

summary_df = pd.DataFrame(rows)
print("\n" + "=" * 100)
print(summary_df.to_string(index=False))
print("=" * 100)
summary_df.to_csv(os.path.join(PLOTS_DIR, "summary_table.csv"), index=False)


In [ ]:
# reshard overhead
# separately from headline carbon for transparency.
TRAINING_W = 500
print("\nReshard overhead (mid-run idle, post tail-trim):")
for name, cfg in configs.items():
    if not cfg["is_schedule"]:
        continue
    P = cfg["df"]["total_power_W"].to_numpy()
    ci_arr = ci_at_wall(len(P))
    warm = int(len(P) * 0.1)
    idle_mask = np.zeros(len(P), dtype=bool)
    idle_mask[warm:] = P[warm:] < TRAINING_W
    n_idle  = idle_mask.sum()
    idle_s  = n_idle * INTERVAL_SECONDS
    idle_g = (P[idle_mask] * INTERVAL_SECONDS / 3_600_000
              * ci_arr[idle_mask]).sum()
    pct = idle_g / cfg["carbon_varying_g"] * 100 if cfg["carbon_varying_g"] else 0
    print(f"  {label(cfg):20s}: ~{idle_s:.0f}s idle, "
          f"~{idle_g:.2f}g ({pct:.1f}% of total carbon)")

# quick result
schedules = [c for _, c in configs.items() if c["is_schedule"]]
statics   = [c for _, c in configs.items() if not c["is_schedule"]]
if schedules and statics:
    h_var = schedules[0]["carbon_varying_g"]
    s_var = min(c["carbon_varying_g"] for c in statics)
    pct = (s_var - h_var) / s_var * 100
    print(f"\nHybrid: {h_var:.2f}g    Best static: {s_var:.2f}g    "
          f"Hybrid {'WINS' if h_var < s_var else 'LOSES TO BEST STATIC'} "
          f"({pct:+.1f}%)")


In [ ]:
# plot power
fig, axes = plt.subplots(1, 2, figsize=(14, 5.4))
for ax, varying in zip(axes, [False, True]):
    for _, cfg in configs.items():
        P = cfg["df"]["total_power_W"].to_numpy()
        n = len(P)
        ci = ci_at_wall(n) if varying else np.full(n, CI_FIXED)
        cum, _ = carbon(P, ci)
        x = np.arange(n) * INTERVAL_SECONDS
        kw = {"linewidth": 2.4, "linestyle": "--"} if cfg["is_schedule"] else {"linewidth": 1.7}
        col = COLORS["schedule"] if cfg["is_schedule"] else None
        ax.plot(x, cum, label=label(cfg), color=col, **kw)
        ax.scatter(x[-1], cum[-1], s=42, zorder=3, color=col)
    if varying:
        annotate_ci_buckets(ax)
        ax.set_title(f"Varying CI (from {os.path.basename(CARBON_CSV)})",
                     fontsize=10, fontweight="bold")
        add_ci_colorbar(fig, ax)
    else:
        ax.set_title(f"Fixed CI ({CI_FIXED} gCO₂/kWh)",
                     fontsize=10, fontweight="bold")
    ax.set_xlabel("Wall-clock time (s)")
    ax.set_ylabel("Cumulative CO₂ (g)")
    ax.legend(fontsize=9, frameon=False, loc="upper left")
plt.suptitle(f"Carbon Comparison — {TRAIN_ITERS} iters, {MODEL_NAME}",
             fontsize=11, fontweight="bold", y=1.00)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "graph_carbon_long.png"),
            dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# plot energy
n_cfg = len(configs)
ncols = min(3, n_cfg); nrows = (n_cfg + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(13, 3.0 * nrows))
axes = np.atleast_1d(axes).flatten()

for ax, (_, cfg) in zip(axes, configs.items()):
    df = cfg["df"]
    x = np.arange(len(df)) * INTERVAL_SECONDS
    y = df["total_power_W"].to_numpy()
    warm = int(len(df) * 0.2)
    color = COLORS["schedule"] if cfg["is_schedule"] else "#1976D2"
    ax.plot(x[:warm], y[:warm], color="#B0BEC5", lw=1.4)
    ax.plot(x[warm:], y[warm:], color=color, lw=1.6)
    steady = df["total_power_W"].iloc[warm:].mean()
    ax.axhline(steady, color="#43A047", ls=":", lw=1)

    if cfg["is_schedule"]:
        steps = df["time_step"].to_numpy()
        for j in range(1, len(steps)):
            if steps[j] < steps[j-1] and steps[j-1] > 0:
                ax.axvline(x[j], color="#FF9800", ls="-.", lw=0.8, alpha=0.7)
        title = "Hybrid Schedule"
    else:
        title = f"TP={cfg['tp']}, PP={cfg['pp']}, DP={cfg['dp']}"
    ax.set_title(title, fontsize=9, fontweight="bold")
    ax.set_xlabel("Wall-clock time (s)", fontsize=8)
    ax.set_ylabel("Power (W)", fontsize=8)
    ax.tick_params(labelsize=7)
    ax.text(0.02, 0.92, f"avg={steady:.0f}W",
            transform=ax.transAxes, fontsize=7, color="#2E7D32")

for ax in axes[n_cfg:]:
    ax.set_visible(False)
fig.suptitle("Power Per Configuration",
             fontsize=11, fontweight="bold", y=1.00)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "graph0_power_trends.png"),
            dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# plot 3 bar
ordered = list(configs.values())
labels  = [label(c) for c in ordered]
cols    = [COLORS[kind(c)] for c in ordered]
edges   = ["#E91E63" if c["is_schedule"] else "white" for c in ordered]
ews     = [2.5 if c["is_schedule"] else 0.5 for c in ordered]
times   = [c["train_time_s"]    for c in ordered]
energ   = [c["total_energy_wh"] for c in ordered]
cfix    = [c["carbon_fixed_g"]  for c in ordered]
cvar    = [c["carbon_varying_g"]for c in ordered]

fig, axs = plt.subplots(1, 3, figsize=(16, 4))

axs[0].bar(labels, times, color=cols, edgecolor=edges, linewidth=ews)
for i, v in enumerate(times):
    axs[0].text(i, v + max(times)*0.01, f"{v:.0f}s",
                ha="center", va="bottom", fontsize=8)
axs[0].set_ylabel("Wall-clock time (s)")
axs[0].set_title("Total Time", fontweight="bold")
axs[0].tick_params(axis="x", labelsize=8, rotation=15)

axs[1].bar(labels, energ, color=cols, edgecolor=edges, linewidth=ews)
for i, v in enumerate(energ):
    axs[1].text(i, v + max(energ)*0.01, f"{v:.1f}",
                ha="center", va="bottom", fontsize=8)
axs[1].set_ylabel("Energy (Wh)")
axs[1].set_title("Total Energy", fontweight="bold")
axs[1].tick_params(axis="x", labelsize=8, rotation=15)

xpos = np.arange(len(labels)); w = 0.38
axs[2].bar(xpos - w/2, cfix, w, color=cols, alpha=0.55,
           edgecolor=edges, linewidth=ews, label=f"Fixed CI ({CI_FIXED})")
axs[2].bar(xpos + w/2, cvar, w, color=cols,
           edgecolor="black", linewidth=0.6, label="Varying CI")
for xi, v in zip(xpos, cvar):
    axs[2].text(xi + w/2, v + max(cvar)*0.01, f"{v:.0f}",
                ha="center", va="bottom", fontsize=7)
axs[2].set_xticks(xpos); axs[2].set_xticklabels(labels, fontsize=8, rotation=15)
axs[2].set_ylabel("Carbon (gCO₂)")
axs[2].set_title("Carbon — Fixed vs Varying CI", fontweight="bold")
axs[2].legend(fontsize=8, loc="upper left")

patches = [
    mpatches.Patch(color=COLORS["dp"],       label="Data Parallel"),
    mpatches.Patch(color=COLORS["mixed"],    label="Mixed (TP+DP)"),
    mpatches.Patch(color=COLORS["schedule"], label="Hybrid Schedule"),
]
fig.legend(handles=patches, fontsize=9, ncol=3,
           loc="upper center", bbox_to_anchor=(0.5, 1.03), frameon=False)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "graph_metrics_overview.png"),
            dpi=150, bbox_inches="tight")
plt.show()